In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['hatch.linewidth'] = 0.2
import numpy as np
import pandas as pd
import pickle
from tqdm.notebook import tqdm
import polars as pl
import xgboost as xgb
print("xgboost version:", xgb.__version__)

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.signal_categories import topological_category_labels, topological_category_colors, topological_category_labels_latex, topological_category_hatches, topological_categories_dic
from src.signal_categories import filetype_category_labels, filetype_category_colors, filetype_category_hatches
from src.signal_categories import del1g_detailed_category_labels, del1g_detailed_category_colors, del1g_detailed_category_labels_latex, del1g_detailed_category_hatches, del1g_detailed_categories_dic, del1g_detailed_category_queries
from src.signal_categories import del1g_simple_category_labels, del1g_simple_category_colors, del1g_simple_category_labels_latex, del1g_simple_category_hatches, del1g_simple_categories_dic, del1g_simple_category_queries
from src.signal_categories import train_category_labels, train_category_labels_latex

from src.file_locations import intermediate_files_location

from src.plot_helpers import make_histogram_plot

from src.ntuple_variables.variables import combined_training_vars

from src.df_helpers import lazy_height


In [ ]:
training = "all_vars"
training_vars = combined_training_vars
reco_categories = train_category_labels
reco_category_labels_latex = train_category_labels_latex

In [ ]:
print("loading all_df.parquet...")
all_df = pl.scan_parquet(f"{intermediate_files_location}/all_df.parquet")
print(f"num events in all_df: {lazy_height(all_df)}")

# this only includes predictions for events passing the preselection used during training
print("loading predictions.parquet...")
pred_df = pl.scan_parquet(f"../training_outputs/{training}/predictions.parquet")

print(f"num events in predictions.parquet: {lazy_height(pred_df)}")

print("merging all_df and predictions.pkl...")
merged_df_no_data_drop = all_df.join(
    pred_df, 
    on=["filetype", "run", "subrun", "event"], 
    how="left"
)

del all_df
del pred_df


In [ ]:
full_pred = merged_df_no_data_drop.filter(
    ~pl.col("filetype").is_in(["data", "isotropic_one_gamma_overlay", "delete_one_gamma_overlay", 
    "numuCC_rad_corrected", "NC_coherent_1g_reweighted"])
)
full_data = merged_df_no_data_drop.filter(pl.col("filetype") == "data")

prob_categories = ["prob_" + cat for cat in reco_categories]
for prob in prob_categories:
    full_pred = full_pred.with_columns(pl.col(prob).fill_null(-1))
    full_data = full_data.with_columns(pl.col(prob).fill_null(-1))

generic_pred_df = full_pred.filter(pl.col("wc_kine_reco_Enu") > 0)
non_generic_pred_df = full_pred.filter(pl.col("wc_kine_reco_Enu") < 0)
del full_pred

num_train_events = lazy_height(generic_pred_df.filter(pl.col("used_for_training") == True))
num_test_events = lazy_height(generic_pred_df.filter(pl.col("used_for_testing") == True))

#num_train_events = generic_pred_df.filter(pl.col("used_for_training") == True).height
#num_test_events = generic_pred_df.filter(pl.col("used_for_testing") == True).height
print(f"num_train_events: {num_train_events}, num_test_events: {num_test_events}")
frac_test = num_test_events / (num_train_events + num_test_events)
print(f"weighting up preselected prediction events by the fraction of test/train events: {frac_test:.3f}")

# Modify weights using polars expressions
generic_pred_df = generic_pred_df.with_columns(
    pl.when(pl.col("used_for_testing"))
    .then(pl.col("wc_net_weight") / frac_test)
    .otherwise(pl.col("wc_net_weight"))
    .alias("wc_net_weight")
)

full_pred = pl.concat([generic_pred_df, non_generic_pred_df])
del generic_pred_df
del non_generic_pred_df

test_pred = full_pred.filter(pl.col("used_for_testing") == True)


In [ ]:
merged_df = pl.concat([test_pred, full_data])

del test_pred
del full_data
presel_merged_df_allvars = merged_df.filter(pl.col("wc_kine_reco_Enu") > 0)

presel_merged_data_df_allvars = presel_merged_df_allvars.filter(pl.col("filetype") == "data")
presel_merged_pred_df_allvars = presel_merged_df_allvars.filter(pl.col("filetype") != "data")


# Preselection Histogram

In [ ]:
load_vars = [
    "filename",
    "filetype",
    "detailed_run_period",
    "run",
    "subrun",
    "event",

    "wc_kine_reco_Enu",
    "wc_match_isFC",

    "normal_overlay",
    "del1g_overlay",
    "iso1g_overlay",
    "wc_truth_inFV",
    "wc_truth_NCDeltaRad",
    "wc_truth_numuCCDeltaRad",
    "wc_true_has_pi0_dalitz_decay",
    "wc_true_has_photonuclear_absorption",
    "true_num_gamma_pairconvert_in_FV",
    "true_num_gamma_pairconvert_in_FV_20_MeV",
    "wc_true_gamma_pairconversion_spacepoint_max_min_distance",
    "wc_truth_0pi0",
    "wc_truth_1pi0",
    "wc_truth_multi_pi0",
    "wc_truth_isNC",
    "wc_truth_numuCC",
    "wc_truth_notnumuCC",
    "wc_truth_nueCC",
    "wc_truth_notnueCC",
    "true_num_prim_gamma",


    "wc_truth_0e",
    "wc_truth_1e",
    "wc_truth_0g",
    "wc_truth_1g",
    "wc_truth_2g",
    "wc_truth_3plusg",
    "wc_truth_0mu",
    "wc_truth_1mu",
    "wc_truth_Np",
    "wc_truth_0p",
    "wc_truth_nuPdg",
    "wc_truth_isCC",

    "wc_weight_cv",
    "wc_weight_spline",
    "wc_net_weight",
]

In [ ]:
presel_merged_df = presel_merged_df_allvars.select(load_vars).collect()


In [ ]:
"""
"flux_all",
"reint_all",

"All_UBGenie",
"AxFFCCQEshape_UBGenie",
"DecayAngMEC_UBGenie",
"NormCCCOH_UBGenie",
"NormNCCOH_UBGenie",
"RPA_CCQE_UBGenie",
"ThetaDelta2NRad_UBGenie",
"Theta_Delta2Npi_UBGenie",
"VecFFCCQEshape_UBGenie",
"XSecShape_CCMEC_UBGenie",
"xsr_scc_Fa3_SCC",
"xsr_scc_Fv3_SCC",
"""

weights_df = pl.read_parquet(f"{intermediate_files_location}/presel_weights_df.parquet")


In [ ]:
presel_detvar_df = pl.scan_parquet(f"{intermediate_files_location}/detvar_presel_df_train_vars.parquet")

presel_detvar_df = presel_detvar_df.select(load_vars + ["vartype"]).collect()


In [ ]:
for plot in ["All", "Run 4", "Run 5"]:

    if plot == "All":
        df = presel_merged_df
        detvar_df = presel_detvar_df
    elif plot == "Run 4":
        df = presel_merged_df.filter(pl.col("detailed_run_period").str.contains("4"))
        detvar_df = presel_detvar_df.filter(pl.col("detailed_run_period").str.contains("4"))
    elif plot == "Run 5":
        df = presel_merged_df.filter(pl.col("detailed_run_period").str.contains("5"))
        detvar_df = presel_detvar_df.filter(pl.col("detailed_run_period").str.contains("5"))
    
    print(f"plotting {plot} Preselection, num events: {df.height}")

    make_histogram_plot(pred_and_data_sel_df=df, weights_df=weights_df, 
                bins=np.linspace(0, 2000, 21), 
                var="wc_kine_reco_Enu", display_var=r"WC Reconstructed $E_\nu$ (MeV)", title=plot + " Preselection",
                include_ratio=True, include_decomposition=False,
                selname="generic_presel",
                dont_load_rw_from_systematic_cache=True, dont_load_detvar_from_systematic_cache=False, 
                use_rw_systematics=True, use_detvar_systematics=True, detvar_df=presel_detvar_df,
                use_detvar_bootstrapping=True, num_bootstrap_rounds_detvar=100, num_bootstrap_samples_detvar=100,
                plot_sys_breakdown=True, 
                )
